# 01 — RF environment demo

What the receiver is up against: 0.5–18 GHz in 128 channels, a 4-channel window,
and eight emitter classes whose activity spans six orders of magnitude in time.

Run top to bottom; takes about 30 s on CPU.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from smartscan.config import load_config
from smartscan.env.rf_environment import build_episode, generate_scenario
from smartscan.hal.simulated import detection_probability_tensor

cfg = load_config("../configs/medium.yaml")
print(f"config hash {cfg.hash()[:12]}")
print(f"B={cfg.n_channels} channels, K={cfg.receiver.ibw_channels} (K/B = 1/{cfg.n_channels // cfg.receiver.ibw_channels})")
print(f"T={cfg.n_slots} slots of {cfg.time.dt_s * 1e3:.0f} ms = {cfg.time.episode_s} s")

## The order of battle

Note the **detectable fraction** column: a circular scanner is visible for only a
few per cent of the episode. That is the needle.

In [ ]:
scenario = generate_scenario(cfg.run.seed, config=cfg)
episode = build_episode(scenario)
pd = detection_probability_tensor(episode, cfg)

print(f"{'class':22}{'ch':>4}{'f (GHz)':>9}{'Ts (s)':>8}{'threat':>8}{'detectable':>12}")
for t in episode.truth:
    frac = float((pd[t.home_channel] > 0.05).mean())
    ts = f"{t.scan_period_s:.2f}" if np.isfinite(t.scan_period_s) else "-"
    print(f"{t.emitter_class:22}{t.home_channel:4d}{t.f_center_hz / 1e9:9.2f}{ts:>8}"
          f"{t.threat_priority:8.2f}{frac * 100:11.1f}%")

## Ground-truth waterfall

Frequency (vertical) against time (horizontal). Continuous emitters are solid
lines; scanning radars are the faint vertical ticks where a beam swept past.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
snr = np.where(episode.occupancy > 0, episode.snr_db, np.nan)
im = ax.imshow(snr, aspect="auto", origin="lower", cmap="viridis",
               extent=[0, cfg.time.episode_s, 0, cfg.n_channels],
               vmin=-10, vmax=35, interpolation="nearest")
ax.set_xlabel("time (s)")
ax.set_ylabel("channel")
ax.set_title(f"Ground-truth occupancy and SNR — {cfg.scenario.difficulty} tier, seed {cfg.run.seed}")
fig.colorbar(im, ax=ax, label="SNR (dB)")
plt.tight_layout()

## One scanning radar, close up

The beam dwell is `(beamwidth / 360) * Ts`. Everything between the ticks is
sidelobe, and mostly below the detection threshold.

In [ ]:
scanners = [t for t in episode.truth if t.emitter_class == "CircularScanRadar"]
if scanners:
    t0 = scanners[0]
    time_s = np.arange(cfg.n_slots) * cfg.time.dt_s
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(time_s, pd[t0.home_channel], lw=0.8)
    ax.set_xlabel("time (s)")
    ax.set_ylabel("P(detect | looking)")
    ax.set_title(f"Channel {t0.home_channel}: Ts = {t0.scan_period_s:.2f} s, "
                 f"beam dwell = {t0.params['beam_dwell_s'] * 1e3:.0f} ms")
    ax.set_ylim(-0.05, 1.05)
    for k in range(1, int(cfg.time.episode_s / t0.scan_period_s) + 1):
        ax.axvline(k * t0.scan_period_s, color="crimson", ls=":", lw=0.8)
    plt.tight_layout()
    print(f"detectable in {(pd[t0.home_channel] > 0.5).mean() * 100:.2f}% of slots")

## Detection is probabilistic, never hard-coded

Swerling I costs ~7 dB against the non-fluctuating case at Pd = 0.9 — the
classic fluctuation loss.

In [ ]:
from smartscan.env.propagation import min_snr_for_pd, p_detect

snr_axis = np.linspace(-20, 40, 400)
fig, ax = plt.subplots(figsize=(9, 4.5))
for n in (1, 4, 16, 256):
    ax.plot(snr_axis, p_detect(snr_axis, n_integrate=n, pfa=1e-4, swerling=1),
            label=f"Swerling I, N={n}")
ax.plot(snr_axis, p_detect(snr_axis, n_integrate=1, pfa=1e-4, swerling=0),
        "k--", label="Swerling 0, N=1")
ax.axhline(0.9, color="grey", lw=0.7)
ax.set_xlabel("SNR (dB)"); ax.set_ylabel("Pd"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Detection curves, Pfa = 1e-4")
plt.tight_layout()

print(f"sensitivity (Pd>=0.9 @ Pfa=1e-3), 1 pulse Swerling 0: {min_snr_for_pd(0.9, 1e-3, 1, 0):6.2f} dB")
print(f"sensitivity (Pd>=0.9 @ Pfa=1e-3), 1 pulse Swerling 1: {min_snr_for_pd(0.9, 1e-3, 1, 1):6.2f} dB")

## Reproducibility

The same seed gives byte-identical tensors; a different seed does not.

In [ ]:
again = build_episode(generate_scenario(cfg.run.seed, config=cfg))
other = build_episode(generate_scenario(cfg.run.seed + 1, config=cfg))
print(f"same seed      : {episode.digest()} == {again.digest()}  -> {episode.digest() == again.digest()}")
print(f"different seed : {other.digest()}  -> differs: {episode.digest() != other.digest()}")